# popcoord Demo

This notebook demonstrates how to use the `popcoord` package to query population statistics, demographics, and density for any location on Earth.

## Installation

First, make sure you have the package installed:

```bash
pip install popcoord          # For API backend (lightweight)
pip install popcoord[raster]  # For both API and raster backends
```

## Import the Package

In [ ]:
import popcoord

## Example 1: Total Population Query

Get the total population within a radius around a coordinate (Amsterdam).

In [ ]:
# Query population within 10 km of Amsterdam
pop = popcoord.population(
    lat=52.38,
    lon=4.90,
    radius_km=10,
    year=2020
)

print(pop)
print(f"\nTotal population: {pop.total:,.0f} people")

## Example 2: Demographics Breakdown

Get detailed age and sex demographics for London.

In [ ]:
# Query demographics within 15 km of London
demo = popcoord.demographics(
    lat=51.51,
    lon=-0.13,
    radius_km=15,
    year=2020
)

print(demo)
print(f"\nTotal: {demo.total:,.0f}")
print(f"Male: {demo.male:,.0f}")
print(f"Female: {demo.female:,.0f}")
print(f"Sex ratio (M/F): {demo.sex_ratio:.3f}")
print(f"Dependency ratio: {demo.dependency_ratio:.3f}")
print(f"Median age bucket: {demo.median_age_bucket}")

### Access Individual Age Groups

In [ ]:
# Print first few age groups
for label in ['0_1', '1_4', '5_9', '20_24', '65_69', '80_plus']:
    ag = demo.age_groups[label]
    print(f"{label:>8s}: total={ag.total:>10,.0f}, male={ag.male:>10,.0f}, female={ag.female:>10,.0f}")

### Full Summary Report

In [ ]:
print(demo.summary())

## Example 3: Population Density

Calculate population density statistics for New York City.

In [ ]:
# Query density within 20 km of NYC (using API backend for speed)
density = popcoord.density(
    lat=40.71,
    lon=-74.01,
    radius_km=20,
    year=2020,
    backend="api"  # Faster, but less detailed than raster
)

print(density)
print(f"\nMean density: {density.mean_density:,.1f} people/km²")
print(f"Max density: {density.max_density:,.1f} people/km²")
print(f"Min density: {density.min_density:,.1f} people/km²")
print(f"Total population: {density.total_population:,.0f}")
print(f"Search area: {density.area_km2:,.1f} km²")

## Example 4: Comparing Different Locations

Let's compare population statistics across multiple major cities.

In [ ]:
cities = {
    "Paris": (48.86, 2.35),
    "Tokyo": (35.68, 139.65),
    "Mumbai": (19.08, 72.88),
    "São Paulo": (-23.55, -46.63),
    "Cairo": (30.04, 31.24)
}

radius = 10  # km
year = 2020

print(f"Population within {radius} km radius (year {year}):\n")
print(f"{'City':<15} {'Latitude':>10} {'Longitude':>10} {'Population':>15}")
print("-" * 55)

for city_name, (lat, lon) in cities.items():
    pop = popcoord.population(lat=lat, lon=lon, radius_km=radius, year=year)
    print(f"{city_name:<15} {lat:>10.2f} {lon:>10.2f} {pop.total:>15,.0f}")

## Example 5: Historical Comparison

Compare population changes over time for a single location.

In [ ]:
# Compare Beijing population from 2000 to 2020
beijing = (39.90, 116.40)
radius_km = 25

print(f"Population within {radius_km} km of Beijing over time:\n")
print(f"{'Year':>6} {'Population':>15} {'Change':>12}")
print("-" * 38)

prev_pop = None
for year in range(2000, 2021, 5):
    pop = popcoord.population(
        lat=beijing[0],
        lon=beijing[1],
        radius_km=radius_km,
        year=year
    )
    
    change = ""
    if prev_pop is not None:
        pct_change = ((pop.total - prev_pop) / prev_pop) * 100
        change = f"+{pct_change:>5.1f}%"
    
    print(f"{year:>6} {pop.total:>15,.0f} {change:>12}")
    prev_pop = pop.total

## Example 6: Using Different Backends

Compare results from the API and raster backends.

> **Note:** The raster backend requires `rasterio` to be installed: `pip install popcoord[raster]`

In [ ]:
# Compare backends for the same query (Sydney)
lat, lon = -33.87, 151.21
radius = 15
year = 2020

# API backend (default, faster)
pop_api = popcoord.population(
    lat=lat, lon=lon, radius_km=radius, year=year, backend="api"
)

print(f"API backend: {pop_api.total:,.0f} people")

# Uncomment if you have rasterio installed:
# pop_raster = popcoord.population(
#     lat=lat, lon=lon, radius_km=radius, year=year, backend="raster"
# )
# print(f"Raster backend: {pop_raster.total:,.0f} people")
# print(f"Difference: {abs(pop_api.total - pop_raster.total):,.0f}")

## Example 7: Exploring All Age Groups

In [ ]:
# Get demographics for Mexico City
demo = popcoord.demographics(
    lat=19.43, lon=-99.13, radius_km=20, year=2020
)

# Create a simple bar chart of population by age group
print("Population by Age Group (Mexico City, 20 km radius):\n")

max_pop = max(ag.total for ag in demo.age_groups.values())

for label, ag in demo.age_groups.items():
    bar_length = int((ag.total / max_pop) * 40)
    bar = "█" * bar_length
    pct = (ag.total / demo.total) * 100
    print(f"{label:>8s} {bar:<40s} {ag.total:>10,.0f} ({pct:>4.1f}%)")

## Parameter Reference

All functions accept these parameters:

- **`lat`** (float): Latitude in WGS-84 decimal degrees. Range: `-90` to `90`
- **`lon`** (float): Longitude in WGS-84 decimal degrees. Range: `-180` to `180`
- **`radius_km`** (float): Search radius in kilometers. Must be positive.
- **`year`** (int): Reference year. Valid range: `2000` to `2020`. Default: `2020`. Values outside this range are automatically clamped.
- **`backend`** (str): Data source backend. Options:
  - `"api"` (default for `population()` and `demographics()`): Uses WorldPop REST API, lightweight, requires only `requests`
  - `"raster"` (default for `density()`): Reads Cloud-Optimized GeoTIFFs, more detailed, requires `rasterio`

### Available Age Groups

When using `demographics()`, the following age group labels are available:
- `0_1`, `1_4`, `5_9`, `10_14`, `15_19`, `20_24`, `25_29`, `30_34`, `35_39`, `40_44`, `45_49`, `50_54`, `55_59`, `60_64`, `65_69`, `70_74`, `75_79`, `80_plus`

## Summary

The `popcoord` package provides a simple interface to query population data:

1. **`population()`** - Get total population count
2. **`demographics()`** - Get age and sex breakdown
3. **`density()`** - Get population density statistics

All powered by WorldPop open data with global coverage from 2000-2020.